## 检查几种医疗图像格式的加载速度（dicom,nii,nii.gz,mhd,mha）
1. 该试验通过`LABELLED_DICOM`文件夹中的数据进行测试
    * 从`LABELLED_DICOM`的dicom数据，处理生成各种格式的数据，并copy到统一的文件夹中，以备后续使用

In [1]:
import os
import sys
import itk
import SimpleITK as sitk
import matplotlib.pyplot as plt
%matplotlib inline
import shutil
import time
import numpy as np

In [2]:
data_in_root = '../data/Liver'
data_out_root = '../data/processed_liver'

# mask_in_series = '../data/Liver/3Dircadb1.1/LABELLED_DICOM/LABELLED_DICOM'
# mask_out_path = '../data/processed_liver/3Dircadb1.1/label'
# os.makedirs(mask_out_path, exist_ok=True)

def generate_different_formats_data(mask_in_series, mask_out_path):
    print('====> begin process {}'.format(mask_in_series))
    # load dicom
    reader = sitk.ImageSeriesReader()
    filenamesDicom = reader.GetGDCMSeriesFileNames(mask_in_series)
    reader.SetFileNames(filenamesDicom)
    dicom_imgs = reader.Execute()

    # copy dicom to new path
    shutil.copytree(mask_in_series, mask_out_path+'/dcm')

    # generate *.nii format
    sitk.WriteImage(dicom_imgs, os.path.join(mask_out_path, 'data.nii'))

    # generate *.nii.gz format
    sitk.WriteImage(dicom_imgs, os.path.join(mask_out_path, 'data.nii.gz'))

    # generate *.mhd format
    sitk.WriteImage(dicom_imgs, os.path.join(mask_out_path, 'data.mhd'))

    # generate *.mha format 
    sitk.WriteImage(dicom_imgs, os.path.join(mask_out_path, 'data.mha'))
    
    # generate *.npy format
    np_img = sitk.GetArrayFromImage(dicom_imgs)
    with open(os.path.join(mask_out_path, 'data.npy'), 'wb') as f:
        np.save(f, np_img)
    print('====> end process {}\n'.format(mask_in_series))

    
# 生成文件，需要重新生成的时候，请解开注释
for sub_root_name in os.listdir(data_in_root):
    sub_root = os.path.join(data_in_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    mask_in_series = os.path.join(sub_root, 'LABELLED_DICOM/LABELLED_DICOM')
    if not os.path.isdir(mask_in_series):
        print('mask label not exist:\t{}'.format(mask_in_series))
        continue
    mask_out_path = '../data/processed_liver/{}/label'.format(sub_root_name)
    os.makedirs(mask_out_path, exist_ok=True)
    generate_different_formats_data(mask_in_series, mask_out_path)

====> begin process ../data/Liver/3Dircadb1.10/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.10/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.20/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.20/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.2/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.2/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.9/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.9/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.5/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.5/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.14/LABELLED_DICOM/LABELLED_DICOM
====> end process ../data/Liver/3Dircadb1.14/LABELLED_DICOM/LABELLED_DICOM

====> begin process ../data/Liver/3Dircadb1.15/LABELLED_DICOM/LABELLED_DICOM
====> end p

# 分别加载不同格式的数据并记录各自的总时间

In [3]:
data_root = '../data/processed_liver'
load_dcm_time = 0
load_nii_time = 0
load_nii_gz_time = 0
load_mha_time = 0
load_mhd_time = 0

load_series_num = 0
load_layers_num = 0

# load dicom
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/dcm')
    if not os.path.isdir(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    load_series_num += 1
    reader = sitk.ImageSeriesReader()
    filenamesDicom = reader.GetGDCMSeriesFileNames(series_uid)
    reader.SetFileNames(filenamesDicom)
    img = reader.Execute()
    imgOrignal = sitk.GetArrayFromImage(img)
    load_layers_num += len(filenamesDicom)
load_dcm_time = time.time() - beg

# load nii
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/data.nii')
    if not os.path.isfile(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    img = sitk.ReadImage(series_uid)
    imgOrignal = sitk.GetArrayFromImage(img)
load_nii_time = time.time() - beg

# load nii.gz
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/data.nii.gz')
    if not os.path.isfile(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    img = sitk.ReadImage(series_uid)
    imgOrignal = sitk.GetArrayFromImage(img)
load_nii_gz_time = time.time() - beg

# load mhd
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/data.mhd')
    if not os.path.isfile(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    img = sitk.ReadImage(series_uid)
    imgOrignal = sitk.GetArrayFromImage(img)
load_mhd_time = time.time() - beg


# load mha
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/data.mha')
    if not os.path.isfile(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    img = sitk.ReadImage(series_uid)
    imgOrignal = sitk.GetArrayFromImage(img)
load_mha_time = time.time() - beg

# load npy
beg = time.time()
for sub_root_name in os.listdir(data_root):
    sub_root = os.path.join(data_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    series_uid = os.path.join(sub_root, 'label/data.npy')
    if not os.path.isfile(series_uid):
        print('{} not exist!'.format(series_uid))
        continue
    with open(series_uid, 'rb') as f:
        np_img = np.load(f)
load_npy_time = time.time() - beg

In [4]:
print('load dicom time elapsed:\t{:.3f}'.format(load_dcm_time))
print('load nii time elapsed:\t{:.3f}'.format(load_nii_time))
print('load nii.gz time elapsed:\t{:.3f}'.format(load_nii_gz_time))
print('load mhd time elapsed:\t{:.3f}'.format(load_mhd_time))
print('load mha time elapsed:\t{:.3f}'.format(load_mha_time))
print('load npy time elapsed:\t{:.3f}'.format(load_npy_time))

print('load dicom nums:\t{}'.format(load_series_num))
print('load series layers total nums:\t{}'.format(load_layers_num))

load dicom time elapsed:	94.872
load nii time elapsed:	69.614
load nii.gz time elapsed:	114.213
load mhd time elapsed:	21.569
load mha time elapsed:	28.596
load npy time elapsed:	10.881
load dicom nums:	20
load series layers total nums:	2823


(260, 512, 512)
